In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {
        "id": "view-in-github",
        "colab_type": "text"
      },
      "source": [
        "<a href=\"https://colab.research.google.com/github/AlfredV12/ML-Sample/blob/main/CreditCardNew.ipynb\" target=\"_parent\"><img src=\"https://colab.research.google.com/assets/colab-badge.svg\" alt=\"Open In Colab\"/></a>"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# ==============================================================================\n",
        "# PROJECT: CREDIT RISK DEFAULT SCORING & CLASS IMBALANCE OPTIMIZATION\n",
        "# Framework: Data Preparation -> Feature Engineering -> SMOTE -> Model Evaluation\n",
        "# =============================================================================="
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Setup and Imports"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "import pandas as pd\n",
        "import numpy as np\n",
        "import matplotlib.pyplot as plt\n",
        "import seaborn as sns\n",
        "import warnings\n",
        "\n",
        "# Scikit-Learn Modules\n",
        "from sklearn.model_selection import train_test_split\n",
        "from sklearn.preprocessing import StandardScaler\n",
        "from sklearn.linear_model import LogisticRegression\n",
        "from sklearn.ensemble import RandomForestClassifier\n",
        "from sklearn.metrics import (\n",
        "    classification_report,\n",
        "    confusion_matrix,\n",
        "    roc_auc_score,\n",
        "    roc_curve,\n",
        "    precision_recall_curve,\n",
        "    recall_score\n",
        ")\n",
        "\n",
        "# Imbalanced-Learn Module for SMOTE\n",
        "from imblearn.over_sampling import SMOTE\n",
        "\n",
        "# Configuration\n",
        "warnings.filterwarnings('ignore')\n",
        "sns.set_theme(style=\"whitegrid\", palette=\"muted\")\n",
        "plt.rcParams['figure.figsize'] = (12, 5)\n",
        "\n",
        "print(\"✅ All dependencies loaded successfully!\")"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## STEP 1: LOAD THE DATA (TRAIN AND TEST)"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "print(\"\\n📂 Loading datasets from local files...\")\n",
        "\n",
        "# Correctly load training data with labels, and test data\n",
        "df_train = pd.read_csv(\"/content/cs-training.csv\", index_col=0)\n",
        "df_train.head()"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## STEP 2: DATA PREPARATION & FEATURE ENGINEERING"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "def preprocess_and_engineer(data):\n",
        "    \"\"\"Applies missing value imputation and feature engineering.\"\"\"\n",
        "    df_clean = data.copy()\n",
        "\n",
        "    # Imputation\n",
        "    df_clean['MonthlyIncome'] = df_clean['MonthlyIncome'].fillna(df_clean['MonthlyIncome'].median())\n",
        "    df_clean['NumberOfDependents'] = df_clean['NumberOfDependents'].fillna(df_clean['NumberOfDependents'].mode()[0])\n",
        "\n",
        "    # Outlier Capping\n",
        "    util_cap = 10.0\n",
        "    df_clean['RevolvingUtilizationOfUnsecuredLines'] = np.where(\n",
        "        df_clean['RevolvingUtilizationOfUnsecuredLines'] > util_cap,\n",
        "        util_cap,\n",
        "        df_clean['RevolvingUtilizationOfUnsecuredLines']\n",
        "    )\n",
        "\n",
        "    # Feature Engineering\n",
        "    df_clean['TotalDelinquencyEvents'] = (\n",
        "        df_clean['NumberOfTime30-59DaysPastDueNotWorse'] +\n",
        "        df_clean['NumberOfTime60-89DaysPastDueNotWorse'] +\n",
        "        df_clean['NumberOfTimes90DaysLate']\n",
        "    )\n",
        "    df_clean['IncomePerCapita'] = df_clean['MonthlyIncome'] / (df_clean['NumberOfDependents'] + 1)\n",
        "    df_clean['EstimatedMonthlyDebtUSD'] = df_clean['DebtRatio'] * df_clean['MonthlyIncome']\n",
        "    df_clean['SevereDelinquencyRatio'] = np.where(\n",
        "        df_clean['TotalDelinquencyEvents'] > 0,\n",
        "        df_clean['NumberOfTimes90DaysLate'] / df_clean['TotalDelinquencyEvents'],\n",
        "        0\n",
        "    )\n",
        "    return df_clean\n",
        "\n",
        "# Apply processing to training data\n",
        "df_train_processed = preprocess_and_engineer(df_train)\n",
        "\n",
        "# Separate Features (X) and Target (y) from labeled training set\n",
        "X = df_train_processed.drop(columns=['SeriousDlqin2yrs'])\n",
        "y = df_train_processed['SeriousDlqin2yrs']\n",
        "\n",
        "# Create a labeled validation set from our training data since cs-test.csv has no labels\n",
        "X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)\n",
        "\n",
        "# Scale features\n",
        "scaler = StandardScaler()\n",
        "X_train_scaled = scaler.fit_transform(X_train)\n",
        "X_test_scaled = scaler.transform(X_test)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## STEP 3: CLASS IMBALANCE HANDLING (SMOTE)"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "print(\"\\n⚖️  Applying SMOTE to balance the training data...\")\n",
        "smote = SMOTE(random_state=42)\n",
        "X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## STEP 4: MODEL TRAINING"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "print(\"\\n⚙️ Training Models...\")\n",
        "\n",
        "# Baseline Logistic Regression\n",
        "lr_model = LogisticRegression(max_iter=1000, random_state=42)\n",
        "lr_model.fit(X_train_resampled, y_train_resampled)\n",
        "y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]\n",
        "\n",
        "# Advanced Random Forest\n",
        "rf_model = RandomForestClassifier(\n",
        "    n_estimators=150, max_depth=12, min_samples_split=10,\n",
        "    class_weight='balanced', random_state=42, n_jobs=-1\n",
        ")\n",
        "rf_model.fit(X_train, y_train)\n",
        "y_pred_rf = rf_model.predict(X_test)\n",
        "y_prob_rf = rf_model.predict_proba(X_test)[:, 1]"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## STEP 5: EVALUATION ON TEST SET & DASHBOARD"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "print(\"\\n\" + \"=\"*50)\n",
        "print(\" RANDOM FOREST CLASSIFICATION REPORT (Test Data)\")\n",
        "print(\"=\"*50)\n",
        "print(classification_report(y_test, y_pred_rf, target_names=['Non-Default (0)', 'Default (1)']))\n",
        "\n",
        "rf_auc = roc_auc_score(y_test, y_prob_rf)\n",
        "lr_auc = roc_auc_score(y_test, y_prob_lr)\n",
        "\n",
        "print(f\"📈 Model ROC-AUC Benchmarks:\")\n",
        "print(f\" - Baseline Logistic Regression AUC: {lr_auc:.4f}\")\n",
        "print(f\" - Advanced Random Forest AUC:      {rf_auc:.4f}\")\n",
        "\n",
        "# Threshold Optimization\n",
        "precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_rf)\n",
        "opt_idx = np.argmin(np.abs(recalls - 0.75)) # Target Recall >= 0.75\n",
        "optimal_threshold = thresholds[opt_idx]\n",
        "y_pred_optimal = (y_prob_rf >= optimal_threshold).astype(int)\n",
        "\n",
        "# --- VISUAL DIAGNOSTICS DASHBOARD ---\n",
        "fig, axes = plt.subplots(1, 3, figsize=(18, 5))\n",
        "\n",
        "# 1. Confusion Matrix\n",
        "cm = confusion_matrix(y_test, y_pred_optimal)\n",
        "sns.heatmap(cm, annot=True, fmt=',d', cmap='Reds', ax=axes[0],\n",
        "            xticklabels=['Non-Default', 'Default'],\n",
        "            yticklabels=['Non-Default', 'Default'], cbar=False)\n",
        "axes[0].set_title(f'Confusion Matrix\\n(Optimized Threshold = {optimal_threshold:.3f})', fontweight='bold')\n",
        "axes[0].set_xlabel('Predicted Risk Class')\n",
        "axes[0].set_ylabel('True Risk Class')\n",
        "\n",
        "# 2. ROC Curve\n",
        "fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)\n",
        "fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)\n",
        "\n",
        "axes[1].plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {lr_auc:.2f})', linestyle='--', color='tab:blue')\n",
        "axes[1].plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {rf_auc:.2f})', linewidth=2.5, color='tab:red')\n",
        "axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)\n",
        "axes[1].set_title('ROC Curve Benchmarking', fontweight='bold')\n",
        "axes[1].set_xlabel('False Positive Rate')\n",
        "axes[1].set_ylabel('True Positive Rate')\n",
        "axes[1].legend(loc='lower right')\n",
        "\n",
        "# 3. Feature Importance\n",
        "importances = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)\n",
        "importances.tail(8).plot(kind='barh', ax=axes[2], color='navy')\n",
        "axes[2].set_title('Top 8 Risk Drivers (Feature Importance)', fontweight='bold')\n",
        "axes[2].set_xlabel('Relative Gini Importance Score')\n",
        "\n",
        "plt.tight_layout()\n",
        "plt.show()"
      ]
    }
  ],
  "metadata": {
    "language_info": {
      "name": "python",
      "version": "3.10.12"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}

NameError: name 'null' is not defined